# 03 — VAR contrainte par le graphe

## Rôle dans le projet
On prédit le **débit de chaque capteur 15 min plus tard (t+1)** avec le même protocole que `02_mlp.ipynb`. La différence : au lieu d'un modèle commun, on apprend **un modèle linéaire par capteur**, qui lit le passé de ses voisins routiers.

Le détail mathématique est dans `docs/var_graphe.tex`.

## Le modèle
Pour le capteur i :

$$\hat y_i(t+1) = \mu_i + \sum_{j \in \mathcal N_K(i)} \sum_{k=0}^{9} \beta_{ij}^{(k)} \, v_j(t-k) + \gamma_i^\top c(t)$$

- $\mathcal N_K(i)$ : les capteurs à au plus K sauts de i dans le graphe (matrice `A`), i compris ;
- $c(t)$ : le calendrier (jour et heure) ;
- les poids sont appris par une **régression Ridge**, qui pénalise les grands poids avec l'hyperparamètre `alpha`.

## Protocole
- Découpage **chronologique** : train = 80 % premiers instants, validation = 20 % suivants, test utilisé **une seule fois**.
- `K` et `alpha` sont choisis sur la validation.
- Métriques : RMSE et MAE, en débit normalisé.
- Référence à battre : la **persistance** (le dernier débit connu).

Pas de `StandardScaler` : les débits sont déjà tous entre 0 et 1, donc les poids se comparent directement.

## Préparation
Imports, puis chargement des données sous forme de tableaux (T, 36, 48) : on a besoin de garder la séparation par capteur, donc on n'utilise pas `build`, qui aplatit tout.

In [ ]:
import sys; sys.path.append("..")
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from src.data import load_raw, train_val_split, DATA_PATH, T1_PATH
from src.make_dataset import main as make_dataset
from src.utils import evaluate, log_result
from src import var_graphe

BLEU, ORANGE, VERT, JAUNE, GRIS = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#898781"

if not T1_PATH.exists():  # dataset non versionné (.gitignore) : on le génère
    make_dataset()

X_tr, Y_tr, X_te, Y_te, A = load_raw(T1_PATH)
(X_a, Y_a), (X_v, Y_v) = train_val_split(X_tr, Y_tr)
print("train", X_a.shape, "| validation", X_v.shape, "| test", X_te.shape)

train (1008, 36, 48) | validation (253, 36, 48) | test (840, 36, 48)


### Fonctions utilitaires
- `scores` compare deux tableaux (T, 36).
- `persistance` renvoie le dernier débit connu, la colonne 9 de X.
- `courbes` trace un graphique en lignes. Les graphiques sont interactifs : survole-les pour lire les valeurs.
- `reglage` teste chaque couple (voisinage, `alpha`), et `meilleur` retient celui qui a la plus petite RMSE. On les réutilise pour l'horizon t+2.
- `voir_reglage` affiche le résultat de `reglage`.

« Tous les capteurs » est le VAR classique, sans contrainte du graphe : on l'obtient en donnant à `fit` un graphe où tout le monde est connecté. Il sert de comparaison, donc `meilleur` ne le retient pas.

In [ ]:
def scores(Y, Y_pred):
    return evaluate(Y.ravel(), Y_pred.ravel())


def persistance(X):
    return X[:, :, 9]


def courbes(x, series, titre, x_titre, y_titre, mode="lines+markers"):
    """`series` associe à chaque nom ses valeurs et sa couleur."""
    fig = go.Figure()
    for serie, (y, couleur) in series.items():
        fig.add_trace(go.Scatter(x=list(x), y=y, name=serie, mode=mode, line=dict(color=couleur, width=2)))
    fig.update_layout(template="plotly_white", title=titre, xaxis_title=x_titre, yaxis_title=y_titre,
                      hovermode="x unified")
    return fig


voisinages = {"capteur seul": (A, 0), "1 saut": (A, 1), "2 sauts": (A, 2),
              "tous les capteurs": (np.ones_like(A), 1)}
alphas = [1e-6, 1e-3, 1e-2, 0.1, 0.3, 1, 3]


def reglage(X_a, Y_a, X_v, Y_v):
    """RMSE de validation pour chaque couple (voisinage, alpha)."""
    rmse = {}
    for voisinage, (graphe, K) in voisinages.items():
        for alpha in alphas:
            models, hoods = var_graphe.fit(X_a, Y_a, graphe, K, alpha)
            rmse[voisinage, alpha] = scores(Y_v, var_graphe.predict(X_v, models, hoods))["RMSE"]
    return rmse


def meilleur(rmse):
    """Meilleur couple (voisinage, alpha) parmi les voisinages issus du graphe."""
    return min([c for c in rmse if c[0] != "tous les capteurs"], key=rmse.get)


def voir_reglage(rmse, Y_v, X_v, titre):
    for voisinage in voisinages:
        print(f"{voisinage:18}", " ".join(f"{rmse[voisinage, a]:.4f}" for a in alphas))
    series = {v: ([rmse[v, a] for a in alphas], c) for v, c in zip(voisinages, [BLEU, ORANGE, VERT, JAUNE])}
    fig = courbes(alphas, series, titre, "alpha", "RMSE (validation)")
    fig.add_hline(y=scores(Y_v, persistance(X_v))["RMSE"], line_dash="dash", line_color=GRIS,
                  annotation_text="persistance")
    fig.update_xaxes(type="log")
    fig.show()


print("persistance (val)", scores(Y_v, persistance(X_v)))

persistance (val) {'MAE': 0.025418882281786657, 'RMSE': 0.03809453642424843}


## Choix du voisinage et de la pénalité
On teste chaque couple (voisinage, `alpha`) sur la validation.

In [ ]:
rmse = reglage(X_a, Y_a, X_v, Y_v)
voir_reglage(rmse, Y_v, X_v, "Sans pénalité, le VAR complet est pire que la persistance")

capteur seul       0.0313 0.0313 0.0313 0.0313 0.0315 0.0323 0.0339
1 saut             0.0306 0.0306 0.0306 0.0305 0.0306 0.0314 0.0327
2 sauts            0.0310 0.0309 0.0308 0.0303 0.0303 0.0309 0.0324
tous les capteurs  0.0398 0.0395 0.0378 0.0332 0.0316 0.0312 0.0322


Configuration retenue : la meilleure RMSE de validation, parmi les voisinages issus du graphe.

In [ ]:
nom, alpha = meilleur(rmse)
graphe, K = voisinages[nom]
models, hoods = var_graphe.fit(X_a, Y_a, graphe, K, alpha)
P_v = var_graphe.predict(X_v, models, hoods)

m = scores(Y_v, P_v)
log_result("var_graphe", nom, f"alpha={alpha}", "val", m)
print(nom, "| alpha =", alpha, "|", m)

2 sauts | alpha = 0.3 | {'MAE': 0.01958387036746864, 'RMSE': 0.030305327006904078}


À retenir de ce réglage :
- Sans pénalité, le VAR complet (0,0398) fait moins bien que la persistance (0,0381) : avec 389 poids par capteur pour 1008 instants, il apprend du bruit.
- La contrainte du graphe et la Ridge jouent le même rôle : limiter ce que le modèle peut apprendre.
- Les voisins n'apportent qu'un gain modeste par rapport au capteur seul (0,0313 → 0,0303).
- Les écarts entre les meilleures configurations sont faibles, sur seulement 2,6 jours de validation : le choix de `K` et `alpha` n'est pas très précis.

## Où le modèle se trompe-t-il ?
Erreur selon l'heure de la journée. Sur la validation, chaque heure ne compte que quelques instants : la courbe est indicative.

In [ ]:
heures = X_v[:, 0, 15:39].argmax(axis=1)
presentes = np.unique(heures)
rmse_heure = lambda P: [scores(Y_v[heures == h], P[heures == h])["RMSE"] for h in presentes]

courbes(presentes, {"VAR graphe": (rmse_heure(P_v), BLEU), "persistance": (rmse_heure(persistance(X_v)), ORANGE)},
        "Erreur selon l'heure", "heure", "RMSE (validation)").show()

Le modèle fait mieux que la persistance à presque toutes les heures, mais l'erreur reste élevée aux moments où le débit change vite, notamment vers 16 h. Attention : ce sont les heures du fichier. Le profil du débit laisse penser qu'elles ne sont peut-être pas l'heure locale, c'est à vérifier avant de parler d'heures de pointe.

Un exemple sur une journée de validation : réalité et prédiction à 15 min. Change `capteur` pour en voir un autre.

In [ ]:
capteur = 4
jour = range(96)

courbes(jour, {"réel": (Y_v[jour, capteur], ORANGE), "prédit": (P_v[jour, capteur], BLEU)},
        f"Capteur {capteur} : 24 heures de validation", "instant de validation (1 pas = 15 min)",
        "débit normalisé", mode="lines").show()

## Lecture du modèle : quels capteurs comptent ?

Le modèle d'un capteur est une somme pondérée. On peut donc regarder ses poids pour voir sur quels capteurs il s'appuie.

Chaque capteur lu a 10 poids, un par débit passé. Pour résumer ça en un seul nombre, on additionne leurs valeurs absolues : c'est l'**influence**. Les valeurs absolues évitent qu'un poids positif et un poids négatif s'annulent.

Une grande barre veut dire : « le modèle s'appuie beaucoup sur ce capteur pour prédire le capteur choisi ». Ce n'est pas une preuve de cause à effet, car les débits de capteurs proches se ressemblent.

### Pour un capteur

In [ ]:
M = var_graphe.influence(models, hoods)

lus = np.flatnonzero(M[capteur])
lien = ["lui-même" if j == capteur else "voisin direct" if A[capteur, j] else "voisin de voisin" for j in lus]

fig = px.bar(x=M[capteur, lus], y=[f"capteur {j}" for j in lus], color=lien, orientation="h",
             color_discrete_map={"lui-même": ORANGE, "voisin direct": BLEU, "voisin de voisin": VERT},
             labels={"x": "influence (somme des |poids|)", "y": "", "color": "lien"},
             title=f"Sur quels capteurs s'appuie le modèle du capteur {capteur} ?")
fig.update_layout(template="plotly_white", yaxis_categoryorder="total ascending")
fig.show()

### Pour tous les capteurs
La carte ci-dessous regroupe ces barres pour les 36 capteurs. **Chaque ligne est le graphique précédent** pour un capteur : la ligne i montre sur quels capteurs s'appuie son modèle.

- Plus la case est foncée, plus l'influence est forte.
- Les cases vides sont interdites par le graphe : ces capteurs sont trop loin.
- La diagonale correspond au capteur qui lit son propre passé.

In [ ]:
fig = px.imshow(np.where(M > 0, M, np.nan), zmin=0, zmax=np.quantile(M[M > 0], 0.95),
                color_continuous_scale=["#cde2fb", BLEU, "#0d366b"],
                labels={"x": "capteur lu (j)", "y": "capteur prédit (i)", "color": "influence"},
                title="Influence de chaque capteur sur les autres")
fig.update_layout(template="plotly_white", height=600, width=680)
fig.show()

Les poids forts se concentrent près de la diagonale et par blocs : des capteurs de numéros proches semblent liés.

### Les liens les plus forts
On liste les 5 plus fortes influences entre deux capteurs différents. `A` est symétrique, mais pas les poids appris : un lien fort dans un seul sens peut suggérer le sens de propagation du trafic. C'est un indice, pas une preuve, car les entrées sont très corrélées.

In [ ]:
sans_diag = M - np.diag(np.diag(M))
for k in np.argsort(sans_diag, axis=None)[::-1][:5]:
    i, j = np.unravel_index(k, M.shape)
    print(f"capteur {i} lit capteur {j} : {M[i, j]:.2f} (sens inverse : {M[j, i]:.2f})")

capteur 9 lit capteur 11 : 0.87 (sens inverse : 0.34)
capteur 12 lit capteur 11 : 0.81 (sens inverse : 0.30)
capteur 35 lit capteur 0 : 0.72 (sens inverse : 0.23)
capteur 35 lit capteur 29 : 0.72 (sens inverse : 0.56)
capteur 19 lit capteur 18 : 0.69 (sens inverse : 0.48)


## Évaluation finale sur le test
Configuration retenue, réentraînée sur train+val, puis évaluée **une seule fois** sur le test.

In [ ]:
models, hoods = var_graphe.fit(X_tr, Y_tr, graphe, K, alpha)
m = scores(Y_te, var_graphe.predict(X_te, models, hoods))
log_result("var_graphe", nom, f"alpha={alpha}", "test", m)
print("persistance", scores(Y_te, persistance(X_te)))
print("VAR graphe ", m)

persistance {'MAE': 0.026574404607449916, 'RMSE': 0.038809529194002074}
VAR graphe  {'MAE': 0.023167564337721627, 'RMSE': 0.0336977161085372}


## Horizon t+2 (30 min)
On recommence tout pour prédire le débit **30 min plus tard**, avec **son propre réglage** de `K` et `alpha` : le meilleur choix n'est pas forcément le même qu'à t+1.

Pas besoin de nouveau dataset : la cible du `.mat` d'origine est justement à t+2 (`DATA_PATH`). X est identique, seule la cible change. Le réglage se fait toujours sur la validation.

In [ ]:
X2, Y2, X2_te, Y2_te, _ = load_raw(DATA_PATH)
(X2_a, Y2_a), (X2_v, Y2_v) = train_val_split(X2, Y2)

rmse2 = reglage(X2_a, Y2_a, X2_v, Y2_v)
voir_reglage(rmse2, Y2_v, X2_v, "Réglage de K et alpha à t+2")

capteur seul       0.0380 0.0380 0.0379 0.0379 0.0378 0.0380 0.0389
1 saut             0.0372 0.0372 0.0372 0.0371 0.0372 0.0376 0.0383
2 sauts            0.0381 0.0381 0.0379 0.0372 0.0370 0.0372 0.0381
tous les capteurs  0.0478 0.0475 0.0454 0.0399 0.0379 0.0371 0.0375


Configuration retenue à t+2, puis évaluation sur la validation et sur le test (réentraînement sur train+val).

In [ ]:
nom2, alpha2 = meilleur(rmse2)
graphe2, K2 = voisinages[nom2]
print(nom2, "| alpha =", alpha2)

models2, hoods2 = var_graphe.fit(X2_a, Y2_a, graphe2, K2, alpha2)
m2 = scores(Y2_v, var_graphe.predict(X2_v, models2, hoods2))
log_result("var_graphe", f"{nom2} t+2", f"alpha={alpha2}", "val", m2)
print("val  persistance", scores(Y2_v, persistance(X2_v)), "| VAR graphe", m2)

models2, hoods2 = var_graphe.fit(X2, Y2, graphe2, K2, alpha2)
m2 = scores(Y2_te, var_graphe.predict(X2_te, models2, hoods2))
log_result("var_graphe", f"{nom2} t+2", f"alpha={alpha2}", "test", m2)
print("test persistance", scores(Y2_te, persistance(X2_te)), "| VAR graphe", m2)

2 sauts | alpha = 0.3
val  persistance {'MAE': 0.03661767441898628, 'RMSE': 0.05589856254155217} | VAR graphe {'MAE': 0.02334128744303028, 'RMSE': 0.036987113716574906}
test persistance {'MAE': 0.03723030638899392, 'RMSE': 0.05454945742095179} | VAR graphe {'MAE': 0.026885276231480862, 'RMSE': 0.03911169527315978}


## Bilan
- **Test à t+1** : RMSE 0,0337, soit 13 % de moins que la persistance (0,0388).
- **Test à t+2** : RMSE 0,0391, soit 28 % de moins que la persistance (0,0545). Plus l'horizon s'allonge, plus le modèle se distingue.
- **Réglage** : refait à t+2, il retient la même configuration qu'à t+1 (2 sauts, `alpha` = 0,3).
- L'erreur est plus élevée sur le test que sur la validation : ces jours sont plus difficiles à prédire.
- **Limites** : modèle linéaire, graphe imposé, mêmes `K` et `alpha` pour tous les capteurs.